# YouTube 视频摘要

## 练习目标（理念）

用 **LLM** 总结 YouTube 视频：先取英文字幕（transcript），再按块调用 ChatGPT 做摘要，最后拼成一篇可读总结——不用把整支视频看完。

## 重要提示（成本）

- 长视频字幕很长，分块后会**多次**调用 API，账单可能升高
- 想省钱可以用免费的 **Ollama** 本地模型替代云端（本笔记本默认走 OpenAI）

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| `load_dotenv` + `OPENAI_API_KEY` | 从 `.env` 读密钥并做格式检查 |
| Chat Completions | `openai.chat.completions.create(...)` |
| system / user | system 定摘要风格；user 放字幕文本 |
| 长文本处理 | `split_text` 按句号切块，再逐块 `summarize_text` |

## 怎么跑

1. 运行安装格（`youtube-transcript-api`、`openai`）
2. 准备 `.env`：含有效的 `OPENAI_API_KEY`
3. 在示例 URL 格改 `video_url`，再依次跑：提取 ID → 拉字幕 → 分块摘要


In [ ]:
# 安装依赖：字幕 API 客户端 + OpenAI SDK（已装过可跳过）
!pip install youtube-transcript-api openai


In [ ]:
# ========== 导入：环境、展示、OpenAI、YouTube 字幕、正则 ==========

# 导入标准库 os：读环境变量里的 API Key
import os

# 导入 requests：HTTP 客户端（本笔记本后续未必用到，保留原导入）
import requests
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进进程环境
from dotenv import load_dotenv
# 从 IPython.display 导入 Markdown/display：在笔记本里漂亮渲染摘要
from IPython.display import Markdown, display

# 从 openai 导入 OpenAI：调用云端 Chat Completions
from openai import OpenAI
# 从 youtube_transcript_api 导入 YouTubeTranscriptApi：按 video_id 拉字幕
from youtube_transcript_api import YouTubeTranscriptApi
# 导入标准库 re：用正则从 URL 里抽出 11 位视频 ID
import re

# 若本格报错：先检查依赖是否装好，再对照同目录 troubleshooting 笔记本


In [ ]:
# ========== 环境变量：加载 .env 并做 API Key 健全性检查 ==========

# 加载 .env；override=True 表示用文件里的值覆盖已有同名环境变量
load_dotenv(override=True)
# 从环境读取 OpenAI 密钥（不要把真实 key 写进笔记本）
api_key = os.getenv('OPENAI_API_KEY')

# ========== 检查钥匙：常见配置错误提前提示（文案保持英文，供程序/对照排查）==========

if not api_key:
    # 完全没读到密钥
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    # 读到了，但前缀不像当前常见的项目密钥格式
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    # 首尾有空格/制表符，容易导致认证失败
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    # 基本检查通过（仍不保证账单/权限一定 OK）
    print("API key found and looks good so far!")


In [ ]:
# ========== 创建 OpenAI 客户端：默认读环境变量 OPENAI_API_KEY ==========

# 无参构造：SDK 会自动使用已加载的 OPENAI_API_KEY
openai = OpenAI()


In [ ]:
# ========== YoutubeVideoID：从完整/短链 URL 解析出 11 位 video_id ==========

class YoutubeVideoID:
    def __init__(self, url):
        # 保存原始 URL，方便调试或后续扩展
        self.url = url
        # 构造时立刻解析；失败会抛 ValueError
        self.video_id = self.extract_video_id(url)

    def extract_video_id(self, url):
        """
        Extracts the YouTube video ID from a given URL.
        Supports both regular and shortened URLs.
        """
        # 正则：匹配 youtube.com/?v=... 或 youtu.be/...，捕获 11 位 ID
        regex = r"(?:https?:\/\/)?(?:www\.)?(?:youtube\.com\/(?:[^\/\n\s]+\/\S+\/|\S*\?v=)|(?:youtu\.be\/))([a-zA-Z0-9_-]{11})"
        # re.match：从字符串开头尝试匹配整段 URL
        match = re.match(regex, url)
        
        if match:
            # group(1)：第一个捕获组，即 video_id
            return match.group(1)
        else:
            # 错误文案保持英文（原逻辑依赖此字符串）
            raise ValueError("Invalid YouTube URL")

    def __str__(self):
        # 打印对象时显示解析结果，便于快速确认
        return f"Video ID: {self.video_id}"


In [ ]:
# ========== 用法示例：换这里的 URL 就能摘要另一支视频 ==========

# 示例视频链接（可改）；须是能公开拿到字幕的视频
video_url = "https://www.youtube.com/watch?v=kqaMIFEz15s"

# 实例化：内部会 extract_video_id
yt_video = YoutubeVideoID(video_url)
# 打印 "Video ID: ..."
print(yt_video)


In [ ]:
# ========== get_transcript：按 video_id + 语言代码拉取字幕并拼成一段文本 ==========

def get_transcript(video_id, language='en'):
    try:
        # 向 YouTube 请求指定语言字幕；默认 'en'（英文）。languages 是优先级列表
        transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=[language])
        # 每条字幕是 dict，取 'text' 字段，用空格拼成一整段纯文本
        return " ".join([item['text'] for item in transcript])
    except Exception as e:
        # 失败时打印错误并返回 None（例如无字幕、语言不存在、网络问题）
        print(f"Error fetching transcript: {e}")
        return None


In [ ]:
# ========== 拉取字幕并看长度：字符数可粗估后续要切几块 ==========

# 用上一步解析出的 video_id 取字幕（默认英文）
transcript_text = get_transcript(yt_video.video_id)
# 打印字幕总字符数；很长时后面 split_text 会切成多块
print(len(transcript_text))


In [ ]:
# ========== summarize_text：用 gpt-4o-mini 对单段文本做简短摘要 ==========

def summarize_text(text):
    try:
        # system prompt 保留英文：定义摘要风格与约束（勿翻译，以免改变行为）
        system_prompts = """
        You are a helpful assistant who provides concise and accurate summaries of text. Your task is to:
        
        - Capture the key points of the content.
        - Keep the summary brief and easy to understand.
        - Avoid summarizing overly lengthy texts or breaking them into excessively short summaries.
        - Use bullet points where appropriate to enhance clarity and structure.
        """
        # Chat Completions：system 定规则，user 放待摘要正文
        response = openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompts},
                # user 字符串前缀保留英文；text 为字幕块
                {"role": "user", "content": f"Summarize the following text:\n{text}"}
            ],
            # 限制单次生成长度，控制成本与篇幅
            max_tokens=200
        )
        # 取出模型回复正文
        return response.choices[0].message.content
    except Exception as e:
        # API/网络异常时打印并返回 None
        print(f"Error summarizing text: {e}")
        return None


In [ ]:
# ========== split_text：长字幕按字符上限切块，尽量在句号处断开 ==========

def split_text(text, chunk_size=3000):
    """
    Splits large text into smaller chunks based on the given chunk size.
    Ensures that chunks end with a full stop where possible to maintain sentence integrity.
    
    :param text: str, the text to be split
    :param chunk_size: int, maximum size of each chunk (default 3000 characters)
    :return: list of str, where each str is a chunk of text
    """
    # chunks：存放切好的字符串片段
    chunks = []
    # 只要剩余文本还长于上限，就继续切
    while len(text) > chunk_size:
        # 在 [0, chunk_size] 窗口内从右找最后一个句号，尽量保持句子完整
        split_point = text.rfind('.', 0, chunk_size + 1)  # +1 to include the period itself if it's at chunk_size
        if split_point == -1:  # No period found within the chunk size
            # 窗口内没有句号：硬切在 chunk_size
            split_point = chunk_size
        
        # 有句号则把句号本身切进当前块；硬切则取前 chunk_size 个字符
        chunks.append(text[:split_point + 1] if split_point != chunk_size else text[:chunk_size])
        # 从切点之后继续处理剩余文本
        text = text[split_point + 1:] if split_point != chunk_size else text[chunk_size:]
    
    # 循环结束后若还有尾巴，strip 后作为最后一块
    if text:
        chunks.append(text.strip())
    
    return chunks

# 对整段字幕切块（默认约 3000 字符一块）
transcript_chunks = split_text(transcript_text)

# ========== 逐块摘要：每一块各调一次 API，再拼成全文摘要 ==========

# summaries：收集每块的摘要字符串
summaries = []
for chunk in transcript_chunks:
    # 对当前块调用 ChatGPT 摘要
    summary = summarize_text(chunk)
    summaries.append(summary)


# 用空格把各块摘要拼成一篇；再在笔记本里用 Markdown 渲染
full_summary = " ".join(summaries)
display(Markdown(full_summary))
